In [ ]:
from mp_api.client import MPRester

MP_ID = ''

path_to_folder = 'data-folder'

In [ ]:
def generate_POSCAR(path_to_folder, doc, attributes=None):
    """A folder is created, and a POSCAR file is written from the data extracted from Materials Project.

    Args:
        path_to_folder (str):                            The directory path for storing the generated files.
        doc            (MaterialsProject data structure: The document or structure to generate the POSCAR from.
        attributes     (dict or None, optional):         Attributes to the POSCAR (formation energy, band-gap, etc).

    Returns:
        None
    """

    # Folder with the loaded elements
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)
    
    # Folder with the specific element
    path_to_folder = f'{path_to_folder}/{doc.formula_pretty}'
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)

    # Folder with the specific symmetry of the element
    symmetry_folder = re.sub('/', '-', doc.symmetry.symbol)
    path_to_folder  = f'{path_to_folder}/{symmetry_folder}'
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)
    
    # Writing attributes per atom
    if attributes is not None:
        for key, value in attributes.items():
            print(key, value)
            try:
                np.savetxt(f'{path_to_folder}/{key}', [value])
            except:
                pass
    
    # Writing POSCAR
    with open(f'{path_to_folder}/POSCAR', 'w') as POSCAR_file:
        POSCAR_file.write(f'{doc.formula_pretty} ({doc.material_id})\n')

        POSCAR_file.write('1.0\n')  ### !!! Check this !!! ###

        for i in range(3):
            for j in range(3):
                POSCAR_file.write(f'{doc.structure.lattice.matrix[i, j]}')
                POSCAR_file.write('\t')
            POSCAR_file.write('\n')

        [composition, concentration] = composition_concentration(doc.structure.composition)
        POSCAR_file.write(f'{composition}\n')
        POSCAR_file.write(f'{concentration}\n')

        POSCAR_file.write(f'Direct\n')

        for i in range(len(doc.structure.frac_coords)):
            for j in range(3):
                POSCAR_file.write(f'{doc.structure.frac_coords[i, j]}')
                POSCAR_file.write('\t')
            POSCAR_file.write('\n')

        POSCAR_file.truncate()

In [ ]:
# Earth-abundant, non-metals materials

elements = ['H',  'He', 'Li', 'Be', 'B',  'C',  'N',  'O',  'F',  'Ne', 'Na', 'Mg', 'Al', 'Si', 'P',  'S',  'Cl',
            'Ar', 'K',  'Ca', 'Ti', 'V',  'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se', 'Br',
            'Kr', 'Sr', 'Y',  'Zr', 'Nb', 'Mo', 'Rh', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'I',  'Ba', 'La', 'Ce',
            'Pr', 'Nd', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Yb', 'Lu', 'Hf', 'Ta', 'W',  'Hg', 'Pb', 'Bi',
            'At', 'Rn', 'Fr', 'Ra', 'Pa', 'U',  'Es', 'Fm', 'Md', 'No', 'Lr', 'Rf', 'Db', 'Sg', 'Bh', 'Hs', 'Mt',
            'Ds', 'Rg', 'Cn', 'Nh', 'Fl', 'Mc', 'Lv', 'Ts', 'Og', 'Cs', 'Pd', 'Au', 'Xe']

exclude_elements = ['Po', 'Ac', 'Tc', 'Cf', 'Bk', 'Cm', 'Pu', 'Am', 'Np', 'Pm',
                    'Ir', 'Pt', 'Rb', 'Os', 'Ru', 'Tl', 'Sc', 'Re', 'Th', 'Tm']

fields = ['material_id', 'formula_pretty', 'structure', 'symmetry', 'energy_per_atom', 'band_gap']

for element in elements:
    print(element)
    with MPRester(MP_ID) as mpr:
        docs = mpr.materials.summary.search(elements=[element],
                                  exclude_elements=exclude_elements,
                                  fields=fields)

        for i in range(len(docs)):
            doc = docs[i]
            try:
                if doc.band_gap > 0.5:  # Avoid metals
                    generate_POSCAR(path_to_folder, doc,
                                    attributes={
                                        'EPA':     doc.energy_per_atom,
                                        'bandgap': doc.band_gap
                                        }
                                    )
            except:
                pass